# 02c — Medication features

Identify qualifying recent antibiotic exposure. One row is retained per hospital spell.

In [25]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [26]:
import pandas as pd
from pathlib import Path
from icare_risk.utils.paths import get_path, get_data_root_from_schema
from icare_risk.utils.preprocessing import keep_first_spell

# Define schema configuration to use.
SCHEMA_CFG = 'configs/schema_1000p.yaml'
PHENOTYPES_CFG = 'configs/phenotypes.yaml'

# Paths
DATA_DIR = get_data_root_from_schema(schema_path=get_path(SCHEMA_CFG)) 
DATA_PROC_DIR = DATA_DIR / 'processed'
DATA_PROC_DIR.mkdir(parents=True, exist_ok=True)

# Debug
print(f"Using data from: {DATA_DIR}")

Using data from: /app/data/synthetic/10000p


In [27]:
# --- OPTION A: Load using pandas ---
from icare_risk.clinphen.io.loaders import load_from_pandas

# Load dataframes
dfs = load_from_pandas(
    schema_path=SCHEMA_CFG,
    domains_to_load=['episodes', 'administration'],
    verbose=1
)

✅ Schema validation passed! All files exist and all physical columns match successfully.
Loaded... episodes (2.46MB) shape (10000, 11)
Loaded... administration (105.85MB) shape (175345, 13)


In [30]:
# --- OPTION B: Load using DuckDB ---
from icare_risk.clinphen.io.loaders import load_from_duckdb

# Configure
domains = ['episodes', 'administration']

# Load
dfs = load_from_duckdb(
    schema_path=SCHEMA_CFG,
    phenotype_path=PHENOTYPES_CFG,
    domains_to_load=domains,
    custom_codes={}, skip_mapping=True, 
    case_sensitive=False, verbose=1
)

# Assign
df_episodes = keep_first_spell(dfs['episodes'])
df_administration = dfs['administration']

✅ Schema validation passed! All files exist and all physical columns match successfully.
Skipping 'definitions': missing module or function definition.
Loaded... episodes (2.46MB) shape (10000, 11)
Loaded... administration (103.56MB) shape (175345, 13) | Filtered codes: None
episodes: 10,000 rows -> 10,000 unique spells


In [31]:
# Libraries
from icare_risk.clinphen.utils import get_qualifying_last_abx

target_abx = [
    "co-amoxiclav", "ciprofloxacin", "ceftriaxone",
    "ertapenem", "meropenem",
]

df_medications = get_qualifying_last_abx(
    df_episodes=df_episodes,
    df_admin=df_administration,
    target_abx=target_abx,
    subject_col="SUBJECT",
    min_duration_hours=24,
    verbose=True,
)
df_medications = keep_first_spell(df_medications, "medication features")
display(df_medications.head())


--- Qualifying Last Antibiotic Summary (> 24h) ---
  Qualifying ABX Events :       0 / 10,000 (0.0%)

  No qualifying antibiotics identified.
--------------------------------------------------
medication features: 10,000 rows -> 10,000 unique spells


,SUBJECT,SPELL_IDENTIFIER,ENCNTR_ID,EPISODE_IDENTIFIER,ADMISSION_DATE,ADMISSION_TIME,AGE_AT_ADMISSION,DISCHARGE_DATE,MAIN_SPECIALTY_CODE,MAIN_SPECIALTY_CODE_DESC,INDEX_OF_MULTIPLE_DEPRIVATION_DECILE,qualifying_last_abx
0,19491,4180691,8395002,5415363,2020-01-01,03:49:00,83,2020-01-24,300,general medicine,6,NaN
1,12471,2863060,5915325,7580619,2020-01-01,06:06:00,54,2020-01-14,320,cardiology,6,NaN
2,10286,8211085,9699216,7036930,2020-01-01,07:08:00,91,2020-01-06,140,oral surgery,5,NaN
3,19324,4394625,4840743,1530791,2020-01-01,07:14:00,95,2020-01-21,301,gastroenterology,none,NaN
4,19153,3642717,5138658,4800743,2020-01-01,13:59:00,38,2020-01-06,140,oral surgery,none,NaN


In [32]:
filename = "02d_derived_features_medications.parquet"
output_path = DATA_PROC_DIR / filename
df_medications.to_parquet(output_path, index=False)
print(f"Saved: {output_path}")

Saved: /app/data/synthetic/10000p/processed/02d_derived_features_medications.parquet
